# 7. LLM Churn Explanations (SHAP → LLM)

This notebook turns the model's per-user SHAP explanations into **plain-language churn reports for business users** — e.g. "This user is at high risk of churning, mainly because they recently cancelled auto-renew and their listening activity has dropped."

**Design (why it's built this way):**

```
Trained model (XGBoost)
      │
      ▼
[1] Per-user SHAP        -> how much each feature pushed churn probability up/down, for ONE user
      │
      ▼
[2] Structured extractor -> turn the raw SHAP array into clean, ranked "evidence"
      │                      (feature, value, shap_contribution, direction), keep top-N drivers
      ▼
[3] Prompt builder       -> combine the evidence with a business-meaning dictionary for each feature
      │
      ▼
[4] LLM generation       -> LLM writes the natural-language report + suggested retention actions
      │
      ▼
Business-readable churn report
```

**Key point:** the LLM never invents *why* a user churns. Every factual claim (which features matter, in which direction) comes from SHAP. It's a deterministic, mathematically grounded attribution method. The LLM only does the translation into readable language. This avoids hallucinated explanations and is a responsible way to apply an LLM on top of a model.

**No API key needed.** Report generation uses a small free open-source model that runs right in Colab. If you'd rather use a paid API (Claude/GPT) for higher-quality text later, it's a one-line
swap (see the note in Section 5).

**No retraining.** This notebook loads the winner model (`final_model.pkl`, XGBoost trained on the full train+val data) that `5_modeling` already saved to Drive, so nothing is retrained here.

**Why compute SHAP again if `5_modeling` already did?** That notebook computed *global* SHAP that are values over a 20k-row sample, used to plot which features matter overall. Here we need *per-user* SHAP: for one specific customer, which features pushed *their* churn probability up or down, and by how much. That's what turns into their individual report. It's cheap: we only explain a handful of users at a time.


## 1. Setup & Load the Trained Model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
try:
    import shap
except ImportError:
    %pip install -q shap
    import shap

import numpy as np
import pandas as pd
import os
import json
import joblib

# We generate the reports with a FREE, open-source model that runs right in Colab (no API key, no signup, no cost). transformers + torch handle this.
try:
    import torch
    from transformers import pipeline
except ImportError:
    %pip install -q transformers torch
    import torch
    from transformers import pipeline

%pip install -q --upgrade jinja2


In [ ]:
# --------------------------------------------------------------------
# Load everything we need from Drive: the winner model + metadata saved by 5_modeling (trained on the FULL data, on train+val), plus the test set.
# No retraining happens here; we just load the finished model.
# --------------------------------------------------------------------
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/Churn Prediction/Data/"
RESULTS_DIR = os.path.join(DATA_DIR, "new_data_model_results")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")


In [ ]:
# The winning model, already trained on the full train+val data.
final_model = joblib.load(os.path.join(RESULTS_DIR, "final_model.pkl"))

# Metadata tells us the exact feature list/order the model expects.
with open(os.path.join(RESULTS_DIR, "model_meta.json")) as f:
    meta = json.load(f)
best_model_name = meta["best_model_name"]
FEATURES = meta["features"]
TARGET = meta["target"]

df_test = pd.read_parquet(os.path.join(PROCESSED_DIR, "test.parquet"))
# Match the dtype shrink done in the modeling notebook (keeps SHAP/predict consistent).
for c in df_test.select_dtypes(include=['float64']).columns:
    df_test[c] = df_test[c].astype('float32')

print(f"Loaded model: {best_model_name} (trained on full train+val, no retraining here)")
print(f"Features ({len(FEATURES)}): {FEATURES}")
print(f"Test set: {df_test.shape}")


/usr/lib/python3.12/pickle.py:1760: UserWarning: [19:59:14] WARNING: /__w/xgboost/xgboost/src/collective/../data/../common/error_msg.h:83: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  setstate(state)


Loaded model: XGBoost (trained on full train+val, no retraining here)
Features (9): ['avg_plan_price', 'avg_payment_per_day', 'days_since_first_trans', 'days_since_last_use', 'ratio_auto_renew', 'total_secs_velocity', 'num_unq_velocity', 'last_is_auto_renew', 'last_is_cancel']
Test set: (2606258, 15)


## 2. Feature Dictionary (neutral descriptions only)

The LLM doesn't know what `last_is_auto_renew` measures in our business, so we give it a plain description of each feature. That's all we give it.

**What we deliberately do NOT provide: a story about why the feature drives churn.**

SHAP measures the *model's attribution*, not real-world causation. A SHAP value says "this feature moved the model's prediction up by 0.3 for this user." It does not say "this user will churn because of this feature." Writing pre-baked sentences like "a high price means they're
price-sensitive" would smuggle in a causal claim that SHAP never supported, and for genuinely ambiguous features (is a high `avg_payment_per_day` price sensitivity, or commitment?) we'd just
be inventing an explanation.

So the report says what is actually true: **which factors the model weighted, in which direction, for this specific user.** Any business reading of that stays hedged.


In [ ]:
# Neutral descriptions only: what each feature MEASURES. No claims about why it drives churn.
# The direction (up/down) comes from the sign of the SHAP value, which is a fact about what the
# model did for this user, not a causal claim about the real world.
FEATURE_MEANINGS = {
    'avg_plan_price':
        "the average price of the plans this user has paid for",

    'days_since_first_trans':
        "how long the user has been a customer (days since their first transaction), "
        "bounded by the ~2 years of data we have",

    'avg_payment_per_day':
        "the average amount the user has paid, spread over their entire tenure "
        "(first transaction to last expiration date)",

    'days_since_last_use':
        "how many days since the user last used the service",

    'ratio_auto_renew':
        "the share of the user's transactions that were on auto-renew",

    'total_secs_velocity':
        "the user's recent listening time relative to their longer-term norm: "
        "their 14 day usage divided by their 30 day usage",

    'num_unq_velocity':
        "the user's recent variety of content relative to their longer-term norm: "
        "their 14 day unique-track count divided by their 30 day count",

    'last_is_auto_renew':
        "whether the user's most recent transaction was an auto-renewal (1) or not (0)",

    'last_is_cancel':
        "whether the user's most recent transaction was a cancellation (1) or not (0)",
}

missing = [f for f in FEATURES if f not in FEATURE_MEANINGS]
assert not missing, (
    f"Missing descriptions for: {missing}\n"
    f"Add a plain description of what each feature MEASURES (no causal claims)."
)
print(f"All {len(FEATURES)} model features have descriptions defined.")

All 9 model features have descriptions defined.


## 3. Per-User SHAP Explainer

Build a `TreeExplainer` once (fast for tree models), then a helper that returns the SHAP contribution of every feature for a single user row.


In [ ]:
explainer = shap.TreeExplainer(final_model)

def explain_user(user_row_df):
    """user_row_df: a 1-row DataFrame with the FEATURES columns.
    Returns (shap_contributions dict, base_value, predicted_prob)."""
    shap_vals = explainer.shap_values(user_row_df[FEATURES])
    if isinstance(shap_vals, list):   # some libs return [class0, class1] for binary
        shap_vals = shap_vals[1]
    shap_vals = np.asarray(shap_vals).reshape(-1)  # flatten to 1D over features

    contributions = {feat: float(shap_vals[i]) for i, feat in enumerate(FEATURES)}
    prob = float(final_model.predict_proba(user_row_df[FEATURES])[:, 1][0])
    base = explainer.expected_value
    if isinstance(base, (list, np.ndarray)):
        base = float(np.asarray(base).reshape(-1)[-1])
    return contributions, base, prob


## 4. Structured Evidence Extractor

Rank the features by absolute SHAP impact and keep the top few. For each one we record what it measures, this user's actual value, the SHAP contribution, and **which way the model weighted it for this user**.

Note what's *not* here: any sentence claiming the feature causes churn. The evidence is a factual record of the model's attribution, and it's returned alongside the report so a human can audit that the LLM didn't drift beyond it.


In [ ]:
def build_evidence(user_row_df, contributions, top_n=4):
    """Rank features by absolute SHAP impact and return the top_n as structured evidence.

    We record which direction the MODEL weighted each feature for this user (the sign of the
    SHAP value). We do not assert that the feature causes churn, only that the model's estimate
    moved up or down because of it.
    """
    ranked = sorted(contributions.items(), key=lambda kv: abs(kv[1]), reverse=True)[:top_n]

    evidence = []
    for feat, shap_val in ranked:
        value = user_row_df[feat].iloc[0]
        evidence.append({
            'feature': feat,
            'what_it_measures': FEATURE_MEANINGS[feat],
            'user_value': round(float(value), 3),
            'model_weighted': "upward" if shap_val > 0 else "downward",
            'shap_contribution': round(float(shap_val), 4),
        })
    return evidence

## 5. Prompt Builder + LLM Generation

We hand the LLM the structured evidence and ask it to write a short, business-facing report.

The prompt constrains it hard: use only the given factors, describe **what the model weighted** rather than what causes churn, and hedge any business interpretation ("may suggest" rather than "because"). All the top factors are shown together, so the LLM can note context across them (e.g. a user on auto-renew whose payment level was still weighted upward) without us having to pre-specify every interaction.

Generation uses a **free, open-source model that runs locally in Colab**. (A note below shows how to swap in a paid API like Claude/GPT for higher quality.)


In [ ]:
SYSTEM_PROMPT = """You are a customer-retention analyst assistant for a music streaming service.
You write short, clear churn-risk summaries for the retention team, based ONLY on the model
evidence you are given.

Critical framing: the evidence tells you what the MODEL weighted when scoring this user. It does
NOT establish that those factors cause churn. Write accordingly.

Rules:
- Use ONLY the factors provided. Never invent or assume other factors.
- Say what the model weighted, e.g. "the model weighted their long gap since last use heavily."
  Never say "they are at risk BECAUSE of X."
- If you offer a business interpretation, hedge it: "may suggest", "could indicate". Never state
  it as established fact.
- Where the factors read together (e.g. a recent cancellation alongside a long gap in usage),
  you may note that combination, still hedged.
- Plain business language. Never mention "SHAP", model internals, or raw numbers.
- Then suggest 1-2 retention actions worth trying, framed as worth trying, not as guaranteed fixes.
- About 4-6 sentences total."""

def build_prompt(churn_prob, evidence):
    risk = "high" if churn_prob >= 0.6 else "moderate" if churn_prob >= 0.3 else "low"
    lines = [
        f"The model scored this user's churn probability at {churn_prob:.1%} ({risk} risk).",
        "",
        "The factors the model weighted most heavily, with what each measures, this user's value,",
        "and which way the model weighted it for this user:",
        "",
    ]
    for e in evidence:
        lines.append(
            f"- {e['feature']}: {e['what_it_measures']}. "
            f"This user's value: {e['user_value']}. "
            f"The model weighted this {e['model_weighted']} (toward "
            f"{'higher' if e['model_weighted'] == 'upward' else 'lower'} risk)."
        )
    lines.append("")
    lines.append("Write the churn-risk summary and suggested retention actions.")
    return "\n".join(lines)

In [ ]:
# --------------------------------------------------------------------
# FREE local model — no API key needed. This loads a small instruction-tuned model
# that runs in Colab. First run downloads the model (a minute or two), then it's cached.
#
# We use a compact instruction model so it runs comfortably on Colab's free tier.
# Quality is lower than Claude/GPT, but it's enough to demo the SHAP->LLM report pipeline.
# --------------------------------------------------------------------
GENERATOR_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  # small, free, instruction-tuned

generator = pipeline(
    "text-generation",
    model=GENERATOR_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

def generate_report(churn_prob, evidence):
    prompt = build_prompt(churn_prob, evidence)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    out = generator(
        messages,
        max_new_tokens=300,
        do_sample=False,
    )
    # transformers returns the full chat; grab the assistant's reply text.
    return out[0]["generated_text"][-1]["content"].strip()


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

> **Want higher-quality reports via a paid API instead?** Swap the cell above for an API call.
> For example, with Anthropic (after `%pip install anthropic` and setting an API key via
> `from google.colab import userdata; key = userdata.get('ANTHROPIC_API_KEY')`):
>
> ```python
> import anthropic
> client = anthropic.Anthropic(api_key=key)
> def generate_report(churn_prob, evidence):
>     msg = client.messages.create(
>         model="claude-sonnet-4-6", max_tokens=400,
>         system=SYSTEM_PROMPT,
>         messages=[{"role": "user", "content": build_prompt(churn_prob, evidence)}],
>     )
>     return msg.content[0].text
> ```
>
> Everything else in the notebook stays the same — only `generate_report` changes.
> **Never hardcode a real API key** in a notebook you push to GitHub; use Colab secrets
> (`userdata.get(...)`) as shown.


## 6. The Full Pipeline: one function, user row → report

Ties Sections 3–5 together. Given a user row, it computes SHAP, extracts evidence, and returns both the structured evidence (for auditability) and the LLM's natural-language report.


In [ ]:
def explain_churn_for_user(user_row_df, top_n=4, verbose=True):
    contributions, base, prob = explain_user(user_row_df)
    evidence = build_evidence(user_row_df, contributions, top_n=top_n)
    report = generate_report(prob, evidence)

    if verbose:
        print(f"Predicted churn probability: {prob:.1%}\n")
        print("Structured evidence (from SHAP, auditable):")
        print(json.dumps(evidence, indent=2))
        print("\n" + "=" * 60)
        print("LLM-generated report:\n")
        print(report)

    return {'churn_prob': prob, 'evidence': evidence, 'report': report}


## 7. Demo: explain a few high-risk users

In [ ]:
# Score the test set and pick a few of the highest-risk users to explain.
test_probs = final_model.predict_proba(df_test[FEATURES])[:, 1]
df_test_scored = df_test.copy()
df_test_scored['churn_prob'] = test_probs

high_risk = df_test_scored.sort_values('churn_prob', ascending=False).head(3)

for idx in high_risk.index:
    user_row = df_test.loc[[idx]]
    print("#" * 70)
    result = explain_churn_for_user(user_row)
    print("\n")


[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


######################################################################


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Predicted churn probability: 99.5%

Structured evidence (from SHAP, auditable):
[
  {
    "feature": "last_is_cancel",
    "what_it_measures": "whether the user's most recent transaction was a cancellation (1) or not (0)",
    "user_value": 1.0,
    "model_weighted": "upward",
    "shap_contribution": 3.6844
  },
  {
    "feature": "avg_payment_per_day",
    "what_it_measures": "the average amount the user has paid, spread over their entire tenure (first transaction to last expiration date)",
    "user_value": 0.0,
    "model_weighted": "upward",
    "shap_contribution": 2.5003
  },
  {
    "feature": "days_since_last_use",
    "what_it_measures": "how many days since the user last used the service",
    "user_value": 609.0,
    "model_weighted": "upward",
    "shap_contribution": 1.1064
  },
  {
    "feature": "num_unq_velocity",
    "what_it_measures": "the user's recent variety of content relative to their longer-term norm: their 14 day unique-track count divided by their 30 day cou

[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Predicted churn probability: 99.3%

Structured evidence (from SHAP, auditable):
[
  {
    "feature": "avg_payment_per_day",
    "what_it_measures": "the average amount the user has paid, spread over their entire tenure (first transaction to last expiration date)",
    "user_value": 0.0,
    "model_weighted": "upward",
    "shap_contribution": 2.2964
  },
  {
    "feature": "days_since_last_use",
    "what_it_measures": "how many days since the user last used the service",
    "user_value": 147.0,
    "model_weighted": "upward",
    "shap_contribution": 1.5568
  },
  {
    "feature": "last_is_auto_renew",
    "what_it_measures": "whether the user's most recent transaction was an auto-renewal (1) or not (0)",
    "user_value": 0.0,
    "model_weighted": "upward",
    "shap_contribution": 1.3523
  },
  {
    "feature": "days_since_first_trans",
    "what_it_measures": "how long the user has been a customer (days since their first transaction), bounded by the ~2 years of data we have",
   

## 8. Notes / How to Extend

**Known limitation, stated plainly:** SHAP gives the model's *attribution*, not real-world causation, and it is additive, so it reports each feature's marginal contribution rather than conditional interactions. A tree model may well have learned that a high payment level only matters when auto-renew is off, but an additive attribution can't express that condition. Each SHAP value *does* already account for that user's full feature context (which is why the same value can be weighted up for one user and down for another), but the wording stays at the level of "the model weighted this factor upward." Fully expressing interactions would need SHAP interaction values, which cost O(features squared) to compute.

- **Batch reports:** loop `explain_churn_for_user` over a list of at-risk users to generate a retention worklist. Cache SHAP for the batch (`explainer.shap_values` on the whole subset once) instead of per-row for speed.
- **Structured output:** ask the LLM to return JSON (risk_level, summary, actions) instead of prose if you want to feed the result into a dashboard or CRM.
- **Guardrail:** the structured `evidence` is returned alongside the report, so a human can always audit that the LLM's narrative matches the model's actual attribution.
- **Higher-quality text:** swap the free local model for a paid API (see Section 5); the rest of the pipeline is unchanged.

**One sentence summary:** "Built a churn-prediction pipeline (time-series CV, gradient-boosted trees, SHAP) plus an LLM explanation layer that translates per-user model attributions into business-facing retention reports, using the structured SHAP output as a factual guardrail against hallucinated reasoning."
